#### ◆ Library Imports

The required Python libraries are imported to support the development of the ByT5 joint text restoration task. These libraries provide functionalities for model loading, tokenization, dataset preparation, training configuration and GPU-based computation.

#### ◆ Dataset loading

Training set (`train`), Validation set (`val`), Test set (`test`)

In [ ]:
from utils.dataset_utils import load_dataset_splits

datasets = load_dataset_splits(
    "../../data/dataset_split"
)

train_data = datasets["train"]
val_data = datasets["val"]
test_data = datasets["test"]

In [ ]:
#### ◆ Convert to HuggingFace dataset

In [ ]:
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

#### ◆ Load T5-small

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Model name
model_name = "t5-small"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("T5-small loaded successfully!")

Create the T5 preprocessing function

In [ ]:
max_input_length = 256
max_target_length = 256

def preprocess_function(batch):

    inputs = [
        "normalize: " + text 
        for text in batch["text"]
    ]

    targets = batch["standardized"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

Apply tokenization

In [ ]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True
)
print(tokenized_dataset)

Create the data collator

Same as ByT5:

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Step 5: Define training arguments

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    fp16=True
)

Step 6: Create Trainer

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Step 7: Train

In [ ]:
trainer.train()